# NB04 — Lazo cerrado en el TCLab: controladores de NB03 en la planta real

**Proyecto:** fPINN + sintonía diferenciable de FOPID en el TCLab — *Control Engineering Practice*.
**Secciones del manuscrito:** 5.3 (protocolo de lazo cerrado y métricas) y 6.2–6.4: Tabla 8 (`tab:exp_cl`) y Tabla 9 (`tab:exp_transfer`).

**Qué hace.**
1. Carga los controladores de NB03 (`controladores_NB04.json`: P, B1, B2, B3, B4, todos con Ms ≤ Ms* = 1,6) y la ley
   discreta exportada (FOPID GL con memoria completa, anti-windup por back-calculation, derivada fraccionaria filtrada
   sobre la medición, saturación 0–100 %).
2. Arma un **plan aleatorizado por bloques**: cada repetición corre los 5 controladores en un orden al azar (semilla fija),
   para repartir la deriva del ambiente entre todos.
3. Cada corrida: espera el equilibrio con el ambiente (calentadores apagados) → para cada punto de operación
   T̄ ∈ {35, 50, 65} °C: **aproximación** (r = T̄, hasta asentarse) → **ensayo**, igual al escenario de NB03:
   reinicio sin salto del controlador en el equilibrio, r = T̄ + 5 °C en t = 10 s y Q2 = 0 → 40 % en t = 900 s, 900 s más.
   La aproximación entre puntos es el cambio de punto de operación (sec. 6.3).
4. Calcula por tramo IAE/ITAE de referencia y perturbación, TV(u), energía E = Σ α1 Q1 h, sobrepico, tiempo de
   asentamiento; agrega por corrida (suma sobre los puntos) y saca media ± desviación sobre repeticiones (Tabla 8),
   con Ms y PM de NB03 sobre el modelo identificado en el kit (físico entero OE-LS); ΔJ sustituto → planta, tiempo de
   sintonía y evaluaciones (Tabla 9); pruebas pareadas P frente a cada comparador.

**Modo `HW`.** `'kit'`: el TCLab real (tiempo real, h = 1 s). `'sim'`: **sin hardware**, una planta virtual (por defecto
el físico entero OE-LS de NB02, con ruido 0,03 °C, resolución 0,032 °C y PWM de 8 bits) que corre tan rápido como
puede; sirve para probar el notebook y para ver de antemano qué dará el kit. En `'sim'` hace falta además la carpeta
`NB02_out/kit/` o el zip de NB02 (para el modelo de la planta virtual).

**Duración en el kit.** Una corrida ≈ 2–2,5 h (enfriamiento 15–30 min + 3 × [aproximación 10–25 min + ensayo 30 min]).
Plan completo: 5 controladores × 3 repeticiones = 15 corridas ≈ **35–40 h** de kit. Se puede repartir en varias
sesiones: cada corrida terminada se guarda en `NB04_out/kit/corridas/` y **se salta** al volver a ejecutar;
`MAX_RUNS_SESSION` limita cuántas corridas hace una ejecución. Una corrida cortada se guarda aparte y se repite entera.

**Salidas.** `NB04_out/<HW>/` y `NB04_<HW>_resultados_<fecha>.zip` con todo (CSV por corrida, métricas, tablas en CSV y
filas LaTeX, pruebas pareadas, figuras, metadatos). No requiere PyTorch.

In [ ]:
import os
# Windows/Anaconda: evita que el kernel muera por OpenMP duplicado (numpy/scipy/torch). Debe ir antes de importar numpy.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import json, time, glob, zipfile, datetime, platform, math, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import fsolve
from scipy import stats
from scipy.signal import butter, filtfilt

HW = 'sim'                 # 'kit' (TCLab real) | 'sim' (planta virtual, sin hardware)
MODE = 'quick'             # 'quick' (prueba corta) | 'full' (protocolo completo del manuscrito)
PORT = 'COM3'              # puerto del Arduino; None = detección automática

# ------------------------------------------------------------------------------------------------------------------
# RUTAS. None = buscar en esta carpeta y en '..': NB03_out/kit/ (o el NB03_kit_resultados_*.zip más reciente que no sea
# _quick); NB02_out/kit/ (o el NB02_kit_resultados_*.zip, prefiere _ABC). También se puede poner la carpeta o el .zip.
NB03_DIR = None
NB02_DIR = None            # solo hace falta en HW = 'sim' (planta virtual) y para el chequeo de la sección 3
SIM_PLANT = ('ABC_const_s0', 'PhysInt_OE')   # planta virtual de 'sim': (JSON de surrogates/, modelo)
# ------------------------------------------------------------------------------------------------------------------

CTRL_IDS = ['P', 'B1', 'B2', 'B3', 'B4']     # los de la Tabla 8 (también se pueden agregar 'A1_PID', 'A2', ...)
N_REP = 3                  # repeticiones por controlador
ORDER_SEED = 2026          # semilla del orden aleatorio (bloques = repeticiones)
MAX_RUNS_SESSION = None    # p. ej. 3 = hacer como mucho 3 corridas en esta ejecución; None = todas las pendientes

TS = 1.0                   # periodo de muestreo [s] (= h de NB03)
SC = dict(ops=[35.0, 50.0, 65.0],  # puntos de operación T̄_j [°C] (los de NB03)
          dr=5.0,          # escalón de referencia [°C], en t = T_STEP_R del ensayo
          dq2=40.0,        # escalón de perturbación en Q2 [%], en t = t_r
          t_r=900, t_d=900)  # duración de los tramos de referencia y perturbación [s] (= NB03)
T_STEP_R = 10              # instante del escalón de referencia dentro del ensayo [s] (= NB03)

# Aproximación a cada punto (r = T̄, Q2 = 0) hasta asentarse:
SETTLE = dict(t_min=600,   # duración mínima [s]
              t_max=1500,  # duración máxima [s]; si se agota se avisa y el ensayo arranca igual
              win=120,     # ventana de juicio [s]
              e_tol=0.15,  # |media de e| en la ventana [°C]
              e_max=0.4,   # máx |e| en la ventana [°C]
              T2_tol=0.3)  # variación de T2 en la ventana [°C] (deja enfriar el calentador 2 tras la perturbación)
RESET_AT_TEST = True       # reinicio sin salto del FOPID al empezar cada ensayo (u0 = media de u, y0 = media de y en
                           # los últimos 30 s): el ensayo arranca como en NB03 (equilibrio, memoria vacía)

# Equilibrio con el ambiente antes de cada corrida (igual que NB01)
T_AMB = None               # None = se mide con el kit frío en la sección 4 (o se toma la de NB01)
EQ = dict(window=120, tol=0.2, amb_tol=1.5, t12_tol=1.0, max_wait=3600)
T_MAX = 85.0               # seguridad: si T1 o T2 pasan de esto se aborta y se apaga todo [°C]
# Corte del USB (p. ej. SerialException / PermissionError 13 'El dispositivo no reconoce el comando'):
# se reconecta (hasta `tries` intentos separados `wait_s` s), se apagan los calentadores, la corrida cortada se guarda
# aparte y se repite entera hasta `retry_run` veces; si no se puede reconectar, se detiene todo.
RECONNECT = dict(tries=10, wait_s=30, retry_run=1)
KEEP_AWAKE = True          # Windows: impide que el PC se suspenda mientras corre el notebook (la pantalla sí puede apagarse)

# Métricas
ALPHA1_W = 0.01            # potencia del calentador 1 por % [W/%] (1 W a 100 %, APMonitor) para E = Σ α1 Q1 h
BAND = 0.25                # banda para el tiempo de asentamiento [°C] (> ruido + resolución del sensor)
MS_MODEL = 'PhysInt_OE'    # Ms y PM de la Tabla 8: los de NB03 sobre este modelo identificado en el kit
TF_OVERRIDE = None         # None = Tf de NB03 (5 s). Cambiarlo cambia los controladores: solo para pruebas aparte
# Las métricas de NB03 usan y sin ruido. En el kit, el ruido (σ ≈ 0,03 °C) solo ya suma ≈ 0,024 °C × 900 s ≈ 22 °C·s
# de IAE por tramo, del orden del IAE_d de P (≈ 12). Por eso las métricas se calculan sobre T1 filtrada fuera de línea
# (Butterworth de orden 2, fase cero, filtfilt, sobre la corrida entera); también se guardan las métricas con T1 cruda.
METRIC_FC = 0.05           # frecuencia de corte del filtro de las métricas [Hz] (ωc de los lazos ≈ 0,06 rad/s ≈ 0,01 Hz)

# Planta virtual (solo HW = 'sim')
SIM = dict(noise=0.03, res=0.032, pwm_bits=8, seed=7, amb_sd=0.3, amb_drift_C_per_h=0.0)  # amb_sd: ambiente ±σ por corrida

if MODE == 'quick':
    SC.update(ops=[35.0, 50.0])            # subconjunto de los puntos de NB03 (para poder calcular ΔJ)
    SETTLE.update(t_min=300, t_max=900)
    N_REP = 2
    if HW == 'kit':                        # prueba corta del hardware: 1 controlador, 1 punto, tramos de 300 s
        CTRL_IDS, N_REP = ['B4'], 1
        SC.update(ops=[35.0], t_r=300, t_d=300); SETTLE.update(t_min=300, t_max=600)

OUT_DIR = os.path.join('NB04_out', HW)
RUN_DIR = os.path.join(OUT_DIR, 'corridas')
os.makedirs(RUN_DIR, exist_ok=True)
if HW == 'kit':
    import tclab
print(f'HW = {HW} | MODE = {MODE} | controladores {CTRL_IDS} × {N_REP} rep. | puntos {SC["ops"]} | '
      f'tramos {SC["t_r"]} + {SC["t_d"]} s | salida {OUT_DIR}')

## 1. Controladores de NB03

`controladores_NB04.json` trae, por controlador, p = (Kp, Ki, Kd, λ, μ, Kt), Tf, h y la ley discreta. De la misma
carpeta se leen `metricas_por_op.csv` (métricas de cada controlador en el sustituto, para ΔJ), `resumen.csv` (Ms y PM
sobre cada modelo) y `metadatos_nb03.json` (pesos del costo, escenario).

In [ ]:
def _find(sub, pat, must, prefer=None, user=None):
    """Carpeta que contiene `must` (sub en esta carpeta o en '..', o la indicada); si no, el zip más reciente."""
    has = lambda d: os.path.exists(os.path.join(d, must))
    if user and os.path.isdir(user):
        for d in [user, os.path.join(user, sub)]:
            if has(d): return d, d
        raise FileNotFoundError(f'{user} no contiene {must}')
    if not user:
        for d in [sub, os.path.join('..', sub)]:
            if has(d): return d, d
    if user and user.endswith('.zip'): zips = [user]
    else:
        zips = sorted(glob.glob(pat) + glob.glob(os.path.join('..', pat)), key=os.path.getmtime)
        zips = [z for z in zips if not z.endswith('_quick.zip')] or zips
        if prefer: zips = [z for z in zips if prefer in os.path.basename(z)] or zips
    if zips:
        z = zips[-1]; dest = os.path.join('NB04_out', 'desde_zip', os.path.basename(z)[:-4])
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        for root, _, files in os.walk(dest):
            if os.path.exists(os.path.join(root, must)): return root, z
    return None, None

NB03_PATH, NB03_SRC = _find(os.path.join('NB03_out', 'kit'), 'NB03_kit_resultados_*.zip', 'controladores_NB04.json',
                            user=NB03_DIR)
if NB03_PATH is None: raise FileNotFoundError('No se encontró NB03_out/kit/ ni un zip de NB03. Poner NB03_DIR.')
C03 = json.load(open(os.path.join(NB03_PATH, 'controladores_NB04.json')))
MET03 = pd.read_csv(os.path.join(NB03_PATH, 'metricas_por_op.csv'))
RES03 = pd.read_csv(os.path.join(NB03_PATH, 'resumen.csv'))
META03 = json.load(open(os.path.join(NB03_PATH, 'metadatos_nb03.json'))) if os.path.exists(
    os.path.join(NB03_PATH, 'metadatos_nb03.json')) else {}
COST = META03.get('cost', dict(w_r=1.0, w_d=1.0, w_u=0.2))
if META03.get('mode') == 'quick': print('AVISO: los controladores de NB03 son de una corrida quick')
CTRLS = {k: C03['controllers'][k] for k in CTRL_IDS}
NAMES = ['Kp', 'Ki', 'Kd', 'lam', 'mu', 'Kt']
print('NB03 tomado de:', NB03_SRC, '| diseño:', C03['design'], '| Ms* =', C03['Ms_max'])
print('Ley:', C03['law'])
print(pd.DataFrame({k: {**v['p'], 'Tf': v['Tf']} for k, v in CTRLS.items()}).T.round(4).to_string())
sc03 = META03.get('sc', {})
for key in ['dr', 'dq2', 't_r', 't_d']:
    if key in sc03 and sc03[key] != SC[key]: print(f'AVISO: {key} = {SC[key]} distinto de NB03 ({sc03[key]}): ΔJ no comparable')

## 2. FOPID discreto (la ley de NB03, en numpy y en tiempo real)

Por muestra k desde el último reinicio (u0, y0):
e_k = r_k − y_k ; ẽ_k = e_k + Kt (u_k−1 − v_k−1) ; I_k = h^λ Σ_{i=0..k} w_i^(−λ) ẽ_k−i ;
D_k = h^−μ Σ_{i=0..k} w_i^(μ) (y_k−i − y0) ; d_k = a d_k−1 + (1 − a) D_k, a = Tf/(Tf + h) ;
v_k = u0 + Kp e_k + Ki I_k − Kd d_k ; u_k = sat_[0,100](v_k). Memoria completa (un producto punto de ≤ 10⁴ términos
por muestra: microsegundos).

In [ ]:
def gl_w(a, n):
    return np.cumprod(np.r_[1.0, 1.0 - (a + 1.0) / np.arange(1, n)])

class FOPID:
    def __init__(self, p, Tf, h=TS, n0=4096):
        self.p = dict(p); self.Tf, self.h = float(Tf), float(h)
        self.a = self.Tf / (self.Tf + self.h); self._alloc(n0); self.reset(0.0, 0.0)

    def _alloc(self, n):
        self.n = n; self.wI = gl_w(-self.p['lam'], n); self.wD = gl_w(self.p['mu'], n)
        E, Y = np.zeros(n), np.zeros(n)
        if hasattr(self, 'E'): E[:len(self.E)], Y[:len(self.Y)] = self.E, self.Y
        self.E, self.Y = E, Y

    def reset(self, u0, y0):
        self.u0, self.y0, self.k = float(u0), float(y0), 0
        self.E[:] = 0; self.Y[:] = 0; self.u_prev = self.v_prev = float(u0); self.d = 0.0

    def step(self, r, y):
        p, k = self.p, self.k
        if k >= self.n: self._alloc(2 * self.n)
        e = r - y; et = e + p['Kt'] * (self.u_prev - self.v_prev)
        self.E[k] = et; self.Y[k] = y - self.y0
        I = self.h ** p['lam'] * np.dot(self.wI[:k + 1], self.E[k::-1])
        Dm = self.h ** (-p['mu']) * np.dot(self.wD[:k + 1], self.Y[k::-1])
        self.d = self.a * self.d + (1 - self.a) * Dm
        v = self.u0 + p['Kp'] * e + p['Ki'] * I - p['Kd'] * self.d
        u = min(max(v, 0.0), 100.0)
        self.u_prev, self.v_prev, self.k = u, v, k + 1
        return u, v

def make_ctrl(cid):
    c = CTRLS[cid]; return FOPID(c['p'], TF_OVERRIDE or c['Tf'], c.get('h', TS))

## 3. Planta virtual (modo `'sim'`) y chequeo contra NB03

La planta virtual es el modelo 2×2 de NB02 (balance de energía con radiación, intercambio entre calentadores y sensor,
derivadas GL de orden α y β, memoria completa), simulado desde el ambiente. Tiene la misma interfaz que `tclab.TCLab`
(`T1`, `T2`, `Q1()`, `Q2()`), con ruido, resolución del sensor y PWM de 8 bits; `advance()` avanza un periodo.

**Chequeo:** con la planta del diseño de NB03 (fPINN A+B+C), arrancando en equilibrio, el mismo ruido de NB03 y esta ley
en numpy, IAE_r, IAE_d y TV deben coincidir con `metricas_por_op.csv` de NB03 (diferencias numéricas ~1e−6).

In [ ]:
NB02_PATH, NB02_SRC = _find(os.path.join('NB02_out', 'kit'), 'NB02_kit_resultados_*.zip', 'surrogates', prefer='_ABC',
                            user=NB02_DIR)
if NB02_PATH is None and HW == 'sim':
    raise FileNotFoundError('HW = "sim" necesita NB02_out/kit/surrogates (o el zip de NB02). Poner NB02_DIR.')
load_sur = lambda tag: json.load(open(os.path.join(NB02_PATH, 'surrogates', f'{tag}.json')))

class Model2x2:
    """Modelo de NB02/NB03 (numpy). x = (zH1, zH2, zS1, zS2) = desviación respecto a x̄ (equilibrio o ambiente)."""
    def __init__(self, th, phys, Ta):
        self.th = dict(th); self.th.setdefault('Us', self.th['U']); self.phys = dict(phys); self.phys.setdefault('tc', 1.0)
        self.Ta = float(Ta); self.mcp = self.phys['m'] * self.phys['cp']

    def F(self, x1, x2, q1, q2, Ta=None):
        th, p = self.th, self.phys; TaK = (self.Ta if Ta is None else Ta) + 273.15
        es, A, As = p['eps'] * p['sigma'], p['A'], p['As']
        T1, T2 = TaK + x1, TaK + x2
        ex = th['Us'] * As * (T2 - T1) + es * As * (T2**4 - T1**4)
        return (th['U'] * A * (TaK - T1) + es * A * (TaK**4 - T1**4) + ex + th['alpha1'] * q1,
                th['U'] * A * (TaK - T2) + es * A * (TaK**4 - T2**4) - ex + th['alpha2'] * q2)

    def equilibrium(self, Tbar, q2=0.0):
        x1 = Tbar - self.Ta
        sol = fsolve(lambda v: self.F(x1, v[1], v[0], q2), [x1 / 0.75, 0.3 * x1])
        return dict(Tbar=Tbar, u=float(sol[0]), q2=q2, x=np.array([x1, sol[1], x1, sol[1]]))

class ModelSim:
    """Integra el modelo paso a paso desde x̄ (memoria completa en desviaciones, como closed_loop de NB03)."""
    def __init__(self, model, xbar=None, nmax=20000):
        th = model.th; self.m = model; al, be = th['alpha'], th['beta']
        self.ca = TS ** al / (model.mcp * model.phys['tc'] ** (al - 1)); self.cb = (TS / th['tau_s']) ** be
        self.wa, self.wb = gl_w(al, nmax + 2), gl_w(be, nmax + 2)
        self.xbar = np.zeros(4) if xbar is None else np.asarray(xbar, float)
        self.H = np.zeros((nmax + 1, 2)); self.S = np.zeros((nmax + 1, 2)); self.k = 0   # historia (más antigua primero)

    def y(self): return self.m.Ta + self.xbar[2:] + self.S[self.k]          # (T1, T2) sensores, sin ruido

    def advance(self, q1, q2, Ta=None):
        k = self.k; zH, zS = self.H[k], self.S[k]; x = self.xbar + np.r_[zH, zS]
        f1, f2 = self.m.F(x[0], x[1], q1, q2, Ta)
        mH = self.wa[1:k + 2] @ self.H[k::-1]; mS = self.wb[1:k + 2] @ self.S[k::-1]
        self.S[k + 1] = self.cb * (zH - zS) - mS
        self.H[k + 1] = self.ca * np.array([f1, f2]) - mH
        self.k += 1

class SimLab:
    """Planta virtual con la interfaz de tclab.TCLab. advance() avanza TS con las entradas aplicadas."""
    def __init__(self, model, nmax=20000, seed=0):
        self.sim = ModelSim(model, None, nmax); self.q1 = self.q2 = 0.0; self.t = 0.0
        self.rng = np.random.default_rng(SIM['seed'] + seed); self.version = 'planta virtual'
        self.dTa = self.rng.normal(0, SIM['amb_sd']) if seed else 0.0   # ambiente distinto en cada corrida
        self.port = 'sim'; self._y = None
    def _read(self):
        if self._y is None:
            y = self.sim.y() + self.dTa + self.rng.normal(0, SIM['noise'], 2)
            self._y = np.round(y / SIM['res']) * SIM['res'] if SIM['res'] else y
        return self._y
    @property
    def T1(self): return float(self._read()[0])
    @property
    def T2(self): return float(self._read()[1])
    def _pwm(self, q):
        n = 2 ** SIM['pwm_bits'] - 1; return round(min(max(q, 0), 100) / 100 * n) / n * 100
    def Q1(self, q=None):
        if q is not None: self.q1 = self._pwm(q)
        return self.q1
    def Q2(self, q=None):
        if q is not None: self.q2 = self._pwm(q)
        return self.q2
    def LED(self, v=0): pass
    @property
    def T1_true(self): return float(self.sim.y()[0] + self.dTa)
    def advance(self):
        Ta = self.sim.m.Ta + self.dTa + SIM['amb_drift_C_per_h'] * self.t / 3600
        self.sim.advance(self.q1, self.q2, Ta); self.t += TS; self._y = None
    def close(self): pass

# --- chequeo: ley numpy + modelo de diseño = NB03 --------------------------------------------------------------
if NB02_PATH is not None and C03['design'][0] in [os.path.basename(f)[:-5] for f in glob.glob(os.path.join(NB02_PATH, 'surrogates', '*.json'))]:
    jsD = load_sur(C03['design'][0]); MD = Model2x2(jsD['models'][C03['design'][1]], jsD['phys'], jsD['T_AMB_NB01'])
    N = int(sc03.get('t_r', 900) + sc03.get('t_d', 900)); Nr = int(sc03.get('t_r', 900))
    noise = np.random.default_rng(sc03.get('noise_seed', 123)).normal(0, sc03.get('noise', 0.03), N)
    def sim_nb03(cid, Tbar):
        eq = MD.equilibrium(Tbar); sm = ModelSim(MD, eq['x'], N + 2); c = make_ctrl(cid); c.reset(eq['u'], Tbar)
        e_, u_ = np.zeros(N), np.zeros(N)
        for k in range(N):
            y = sm.y()[0]; r = Tbar + (sc03.get('dr', 5.0) if k >= 10 else 0.0)
            u, _ = c.step(r, y + noise[k]); e_[k], u_[k] = r - y, u
            sm.advance(u, eq['q2'] + (sc03.get('dq2', 40.0) if k >= Nr else 0.0))
        return dict(IAE_r=np.abs(e_[:Nr]).sum() * TS, IAE_d=np.abs(e_[Nr:]).sum() * TS,
                    TV=np.abs(np.diff(np.r_[eq['u'], u_])).sum())
    t0 = time.time(); chk = []
    for cid in CTRL_IDS:
        Tb = sc03.get('ops', [50.0])[len(sc03.get('ops', [50.0])) // 2]
        m = sim_nb03(cid, Tb)
        ref = MET03[(MET03.planta == 'diseño') & (MET03.id == cid) & (MET03.op == Tb)].iloc[0]
        chk.append(dict(id=cid, op=Tb, **{f'{k}_NB04': v for k, v in m.items()},
                        **{f'{k}_NB03': ref[k] for k in m}, max_rel_diff=max(abs(m[k] - ref[k]) / abs(ref[k]) for k in m)))
    chk = pd.DataFrame(chk); print(chk.round(4).to_string(index=False))
    print(f'Chequeo ley NB04 = NB03: diferencia relativa máxima {chk.max_rel_diff.max():.1e} ({time.time() - t0:.0f} s)')
    if chk.max_rel_diff.max() > 1e-3: print('AVISO: la ley de NB04 no reproduce NB03; revisar antes de ir al kit')
else:
    print('Chequeo contra NB03 saltado (no se encontró el sustituto de diseño en NB02).')

## 4. Conexión, ambiente y plan aleatorizado

In [ ]:
lab_handle = None
def open_lab(seed=0):
    global lab_handle
    try: lab_handle.close()
    except Exception: pass
    if HW == 'kit':
        lab_handle = tclab.TCLab(port=PORT) if PORT else tclab.TCLab()
    else:
        js = load_sur(SIM_PLANT[0]); lab_handle = SimLab(Model2x2(js['models'][SIM_PLANT[1]], js['phys'], js['T_AMB_NB01']), seed=seed)
    lab_handle.Q1(0); lab_handle.Q2(0)
    return lab_handle

def heaters_off(lab):
    for f in (lambda: lab.Q1(0), lambda: lab.Q2(0), lambda: lab.LED(0)):
        try: f()
        except Exception as e: print('  aviso al apagar:', e)

META_F = os.path.join(OUT_DIR, 'metadatos_nb04.json')
meta = json.load(open(META_F)) if os.path.exists(META_F) else {}
def save_meta(): json.dump(meta, open(META_F, 'w'), indent=1, default=str)

lab = open_lab()
try:
    r = []
    for _ in range(10):
        r.append((lab.T1, lab.T2))
        if HW == 'kit': time.sleep(1)
        else: lab.advance()
    r = np.array(r)
    print(f'T1 = {r[:, 0].mean():.2f} ± {r[:, 0].std():.3f} °C | T2 = {r[:, 1].mean():.2f} ± {r[:, 1].std():.3f} °C')
    if T_AMB is not None: meta.update(T_AMB=float(T_AMB), T_AMB_origen='fijada a mano')
    elif abs(r[:, 0].mean() - r[:, 1].mean()) < EQ['t12_tol']:
        meta.update(T_AMB=float(r.mean()), T_AMB_origen='medida con el kit frío')
    elif 'T_AMB' not in meta:
        print('AVISO: T1 y T2 difieren (kit caliente). Poner T_AMB a mano o esperar a que se enfríe.')
    meta.update(hw=HW, mode=MODE, puerto=getattr(lab, 'port', PORT), firmware=getattr(lab, 'version', '?'),
                so=platform.platform(), python=platform.python_version(), procesador=platform.processor(),
                nb03=str(NB03_SRC), diseño=C03['design'], ctrl_ids=CTRL_IDS, n_rep=N_REP, sc=SC, settle=SETTLE,
                reset_at_test=RESET_AT_TEST, alpha1_W=ALPHA1_W, tf_override=TF_OVERRIDE,
                sim=(dict(SIM, plant=SIM_PLANT, nb02=str(NB02_SRC)) if HW == 'sim' else None))
    if HW == 'kit': meta['tclab'] = tclab.__version__
    save_meta(); print('T_AMB de referencia:', meta.get('T_AMB'))
finally:
    heaters_off(lab)

# Plan: bloque = repetición; dentro de cada bloque, orden aleatorio de los controladores (fijo por la semilla)
rng = np.random.default_rng(ORDER_SEED); plan = []
for rep in range(1, N_REP + 1):
    for cid in rng.permutation(CTRL_IDS):
        plan.append(dict(n=len(plan) + 1, id=str(cid), rep=rep, archivo=f'corrida_{len(plan) + 1:02d}_{cid}_rep{rep}.csv'))
PLAN = pd.DataFrame(plan); PLAN.to_csv(os.path.join(OUT_DIR, 'plan_NB04.csv'), index=False)
PLAN['hecha'] = PLAN.archivo.map(lambda f: os.path.exists(os.path.join(RUN_DIR, f)))
print(PLAN.to_string(index=False))
q_max_est = max(SC['ops']) + SC['dr']
print(f'Temperatura máxima prevista de T1 ≈ {q_max_est:.0f} °C (T_MAX = {T_MAX} °C)')

## 5. Corridas en lazo cerrado (se guardan y retoman)

Cada muestra: leer T1, T2 → controlador → aplicar Q1, Q2 (la lectura va antes de actuar, como en NB01 y NB03).
Se escribe `corrida_XX_<id>_repR.part.csv` en cada muestra y al terminar se renombra a `.csv`. Si se corta (error,
T_MAX, interrupción), el `.part.csv` queda renombrado como `_cortada_<fecha>.csv` en la próxima ejecución y esa corrida
se repite entera. **Ante cualquier error los calentadores se apagan** (`finally`). Si se corta el USB (error de `serial`), el notebook
reconecta, apaga los calentadores, espera el enfriamiento y **repite esa corrida** (`RECONNECT`); en Windows además impide
que el PC se suspenda mientras corre (`KEEP_AWAKE`). Los cortes quedan en `metadatos_nb04.json` → `incidentes`.

In [ ]:
def wait_equilibrium(lab, label):
    if HW == 'sim': return dict(ok=True, espera_s=0.0, T1_0=lab.T1, T2_0=lab.T2)
    t_amb = meta.get('T_AMB'); print(f'[{label}] esperando equilibrio con el ambiente (T_AMB = {t_amb})')
    t0 = time.time(); hist = []
    while time.time() - t0 < EQ['max_wait']:
        now, T1, T2 = time.time(), lab.T1, lab.T2; hist.append((now, T1, T2))
        if max(T1, T2) > T_MAX: raise RuntimeError(f'{max(T1, T2):.1f} °C > T_MAX con calentadores apagados: revisar el kit')
        old = [h for h in hist if h[0] <= now - EQ['window']]
        if old:
            dT = max(abs(T1 - old[-1][1]), abs(T2 - old[-1][2])); d12 = abs(T1 - T2)
            # solo cuenta estar MÁS caliente que el ambiente: si la sala se enfrió (o T_AMB se midió con el kit tibio),
            # el kit estable por debajo de T_AMB ya está en equilibrio y T_AMB se actualiza
            damb = max(T1, T2) - t_amb if t_amb is not None else 0.0
            print(f'\r  {now - t0:5.0f} s  T1={T1:.2f}  T2={T2:.2f}  ΔT={dT:.2f}  |T1−T2|={d12:.2f}  T−Tamb={damb:+.2f}  ', end='')
            if dT < EQ['tol'] and d12 < EQ['t12_tol'] and damb < EQ['amb_tol']:
                if t_amb is None or damb < 0:
                    meta['T_AMB'] = (T1 + T2) / 2; meta['T_AMB_origen'] = 'equilibrio antes de la corrida'; save_meta()
                print(f'\n  equilibrio OK (T∞ ≈ {(T1 + T2) / 2:.2f} °C)')
                return dict(ok=True, espera_s=now - t0, T1_0=T1, T2_0=T2, T_amb_corrida=(T1 + T2) / 2)
        time.sleep(5)
    print('\n  AVISO: no se alcanzó el equilibrio en max_wait; la corrida arranca igual')
    return dict(ok=False, espera_s=time.time() - t0, T1_0=hist[-1][1], T2_0=hist[-1][2])

def settled(buf_e, buf_T2, k_phase):
    if k_phase < SETTLE['t_min']: return False
    w = int(SETTLE['win'] / TS); e = np.array(buf_e[-w:]); t2 = np.array(buf_T2[-w:])
    return abs(e.mean()) < SETTLE['e_tol'] and np.abs(e).max() < SETTLE['e_max'] and t2.max() - t2.min() < SETTLE['T2_tol']

def run_closed_loop(lab, row):
    cid, fname = row.id, os.path.join(RUN_DIR, row.archivo); part = fname.replace('.csv', '.part.csv')
    if os.path.exists(part):
        cut = part.replace('.part.csv', f'_cortada_{datetime.datetime.now():%Y%m%d_%H%M%S}.csv'); os.replace(part, cut)
        print(f'  corrida cortada anterior guardada como {cut}')
    ctrl = make_ctrl(cid); label = f'{row.n:02d} {cid} rep{row.rep}'
    heaters_off(lab); eq = wait_equilibrium(lab, label)
    print(f'[{label}] lazo cerrado: {CTRLS[cid]["desc"]}')
    T1 = lab.T1; ctrl.reset(0.0, T1)                       # arranque desde el ambiente, sin calentar
    k, lags, t_wall0, info = 0, [], time.time(), []
    t_next = time.monotonic()
    f = open(part, 'w'); f.write('t,fase,op,tau,r,T1,T2,Q1,Q2,v,t_real' + (',T1_true' if HW == 'sim' else '') + '\n')

    def sample(phase, op, tau, r, q2):
        nonlocal k, t_next
        if HW == 'kit':
            dt = t_next - time.monotonic()
            if dt > 0: time.sleep(dt)
            lags.append(time.monotonic() - t_next); t_next += TS
        else:
            if k > 0: lab.advance()
        T1, T2 = lab.T1, lab.T2
        if max(T1, T2) > T_MAX: raise RuntimeError(f'T = {max(T1, T2):.1f} °C > T_MAX = {T_MAX} °C: corrida abortada')
        u, v = ctrl.step(r, T1)
        lab.Q1(u); lab.Q2(q2)
        f.write(f'{k * TS:.1f},{phase},{op},{tau * TS:.1f},{r:.3f},{T1:.3f},{T2:.3f},{u:.3f},{q2:.1f},{v:.3f},'
                f'{time.time() - t_wall0:.3f}' + (f',{lab.T1_true:.4f}' if HW == 'sim' else '') + '\n')
        if k % 60 == 0:
            f.flush(); print(f'\r  t={k * TS:6.0f} s  {phase:5s} T̄={op:.0f}  τ={tau * TS:5.0f}  r={r:5.1f}  T1={T1:6.2f}  '
                             f'T2={T2:6.2f}  Q1={u:5.1f}  Q2={q2:4.1f}   ', end='')
        k += 1
        return T1, T2, u

    try:
        for Tbar in SC['ops']:
            # aproximación (cambio de punto de operación)
            be, bT2, bu, by, kp = [], [], [], [], 0
            while True:
                T1, T2, u = sample('aprox', Tbar, kp, Tbar, 0.0)
                be.append(Tbar - T1); bT2.append(T2); bu.append(u); by.append(T1); kp += 1
                if settled(be, bT2, kp): ok = True; break
                if kp >= SETTLE['t_max']: ok = False; print(f'\n  AVISO: T̄={Tbar} sin asentarse en t_max'); break
            info.append(dict(op=Tbar, aprox_s=kp * TS, aprox_ok=ok))
            if RESET_AT_TEST:
                n30 = int(30 / TS); ctrl.reset(np.mean(bu[-n30:]), np.mean(by[-n30:]))
            # ensayo: referencia (+dr en T_STEP_R) y perturbación (Q2 en t_r)
            for tau in range(int((SC['t_r'] + SC['t_d']) / TS)):
                r = Tbar + (SC['dr'] if tau * TS >= T_STEP_R else 0.0)
                q2 = SC['dq2'] if tau * TS >= SC['t_r'] else 0.0
                sample('ensayo', Tbar, tau, r, q2)
        f.close(); os.replace(part, fname)
        lg = np.array(lags) if lags else np.zeros(1)
        meta.setdefault('corridas', {})[row.archivo] = dict(
            **eq, id=cid, rep=int(row.rep), inicio=datetime.datetime.fromtimestamp(t_wall0).isoformat(timespec='seconds'),
            duracion_s=time.time() - t_wall0, muestras=k, T_AMB=eq.get('T_amb_corrida', meta.get('T_AMB')), aproximaciones=info,
            retraso_reloj_medio_s=float(lg.mean()), retraso_reloj_max_s=float(lg.max()))
        save_meta()
        print(f'\n[{label}] lista: {k} muestras, {(time.time() - t_wall0) / 60:.1f} min'
              + (f', retraso máx. del reloj {lg.max():.2f} s' if HW == 'kit' else ''))
    finally:
        if not f.closed: f.close()
        heaters_off(lab)

def keep_awake(on):
    # SetThreadExecutionState: ES_CONTINUOUS | ES_SYSTEM_REQUIRED mientras corre; ES_CONTINUOUS al terminar
    if HW != 'kit' or not KEEP_AWAKE or platform.system() != 'Windows': return
    try:
        import ctypes; ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | (0x00000001 if on else 0))
    except Exception as e: print('  aviso keep_awake:', e)

def is_comm_error(e):
    # errores del puerto serie: serial.SerialException es OSError; un mensaje cortado puede dar ValueError en tclab
    return HW == 'kit' and isinstance(e, (OSError, ValueError)) and 'T_MAX' not in str(e)

def reconnect():
    global lab_handle
    for i in range(1, RECONNECT['tries'] + 1):
        try: lab_handle.close()
        except Exception: pass
        print(f'  reconectando ({i}/{RECONNECT["tries"]}) en {RECONNECT["wait_s"]} s ...'); time.sleep(RECONNECT['wait_s'])
        try:
            lab_new = open_lab(); T1, T2 = lab_new.T1, lab_new.T2
            print(f'  reconectado: calentadores apagados, T1 = {T1:.2f}, T2 = {T2:.2f} °C'); return lab_new
        except Exception as e: print('   falló:', repr(e)[:120])
    raise RuntimeError('No se pudo reconectar con el kit: DESCONECTAR el USB y la fuente del TCLab y revisar el puerto.')

lab = open_lab(); done_now = 0; keep_awake(True)
try:
    for row in PLAN.itertuples():
        if os.path.exists(os.path.join(RUN_DIR, row.archivo)): continue
        if MAX_RUNS_SESSION is not None and done_now >= MAX_RUNS_SESSION:
            print(f'MAX_RUNS_SESSION = {MAX_RUNS_SESSION}: se para aquí; volver a ejecutar para seguir.'); break
        for intento in range(RECONNECT['retry_run'] + 1):
            if HW == 'sim': lab = open_lab(row.n)      # cada corrida virtual arranca desde el ambiente (ruido y T∞ propios)
            try:
                run_closed_loop(lab, row); done_now += 1; break
            except Exception as e:
                if not is_comm_error(e): raise
                print(f'\n¡Se perdió la comunicación con el kit en {row.archivo}! {e!r}')
                meta.setdefault('incidentes', []).append(dict(fecha=datetime.datetime.now().isoformat(timespec='seconds'),
                                                              corrida=row.archivo, intento=intento, error=repr(e)[:300]))
                save_meta(); lab = reconnect()
                print('  la corrida se repite entera' if intento < RECONNECT['retry_run'] else
                      '  sin más reintentos para esta corrida: queda pendiente y se sigue con la próxima')
finally:
    keep_awake(False)
    heaters_off(lab)
    try: lab.close()
    except Exception: pass
PLAN['hecha'] = PLAN.archivo.map(lambda f: os.path.exists(os.path.join(RUN_DIR, f)))
print(f'Corridas completas: {PLAN.hecha.sum()} de {len(PLAN)}')

In [ ]:
# Apagado de emergencia: ejecutar esta celda si algo quedó a medias (interrupción del kernel, error de USB, etc.)
try:
    heaters_off(lab_handle); lab_handle.close()
except Exception:
    if HW == 'kit':
        try:
            _l = tclab.TCLab(port=PORT) if PORT else tclab.TCLab(); _l.Q1(0); _l.Q2(0); _l.close()
        except Exception as e: print('No se pudo conectar para apagar:', e, '-> desconectar USB y la fuente')
print('Calentadores apagados.')

## 6. Métricas por tramo y por corrida

Por ensayo (t medido desde el inicio del ensayo, e = r − y con y = T1 medida y **filtrada fuera de línea** con fase cero, `METRIC_FC`; ver la nota de la sección 0):
IAE_r = h Σ_{t<t_r} |e|, ITAE_r = h Σ_{t<t_r} (t − 10) |e| ; IAE_d = h Σ_{t≥t_r} |e|, ITAE_d = h Σ_{t≥t_r} (t − t_r) |e| ;
TV(u) = Σ |Δu| (desde u0) ; E = Σ α1 Q1 h [J] ; sobrepico [% de dr] ; tiempo de asentamiento (banda ±`BAND`) ;
máx |e| ante la perturbación. Aproximación: IAE_op y duración. Por corrida se suman los puntos de operación
(como en la Tabla 8); J_kit = media sobre puntos de [w_r IAE_r/IAE_r^B4 + w_d IAE_d/IAE_d^B4 + w_u TV/TV^B4] con la
normalización de B4 **en el sustituto de diseño** (la de ΔJ en NB03).

In [ ]:
def seg_metrics(d, ycol='T1f'):
    t = d.tau.values; e = (d.r - d[ycol]).values; u = d.Q1.values; Nr = t < SC['t_r']; Nd = ~Nr
    ts = t[Nr] - T_STEP_R; td = t[Nd] - SC['t_r']
    out_band = np.where((np.abs(e) > BAND) & Nr & (t >= T_STEP_R))[0]
    t_set = (t[out_band[-1]] - T_STEP_R + TS) if len(out_band) else 0.0
    rs = d.r.values[Nr][-1]
    return dict(IAE_r=TS * np.abs(e[Nr]).sum(), ITAE_r=TS * (np.clip(ts, 0, None) * np.abs(e[Nr])).sum(),
                IAE_d=TS * np.abs(e[Nd]).sum(), ITAE_d=TS * (td * np.abs(e[Nd])).sum(),
                TV=np.abs(np.diff(np.r_[u[0], u])).sum(), E_J=ALPHA1_W * u.sum() * TS,
                overshoot_pct=100 * max(0.0, (d[ycol].values[Nr].max() - rs) / SC['dr']), t_settle_r=t_set,
                emax_d=np.abs(e[Nd]).max(), u_sat_pct=100 * np.mean((u <= 0.0) | (u >= 100.0)))

_bf = butter(2, METRIC_FC / (0.5 / TS))
rows, aprox, raw = [], [], []
for row in PLAN.itertuples():
    fn = os.path.join(RUN_DIR, row.archivo)
    if not os.path.exists(fn): continue
    D = pd.read_csv(fn); D['T1f'] = filtfilt(*_bf, D.T1.values)
    for Tbar in SC['ops']:
        dE = D[(D.fase == 'ensayo') & (D.op == Tbar)]; dA = D[(D.fase == 'aprox') & (D.op == Tbar)]
        if len(dE) == 0: continue
        rows.append(dict(n=row.n, id=row.id, rep=row.rep, op=Tbar, **seg_metrics(dE)))
        raw.append(dict(n=row.n, id=row.id, rep=row.rep, op=Tbar, y='T1 cruda', **seg_metrics(dE, 'T1')))
        if 'T1_true' in dE: raw.append(dict(n=row.n, id=row.id, rep=row.rep, op=Tbar, y='T1 real (sim)', **seg_metrics(dE, 'T1_true')))
        aprox.append(dict(n=row.n, id=row.id, rep=row.rep, op=Tbar, aprox_s=len(dA) * TS,
                          IAE_op=TS * np.abs(dA.r - dA.T1f).sum(), T2_ini_ensayo=dE.T2.iloc[0]))
SEG = pd.DataFrame(rows); APX = pd.DataFrame(aprox)
if len(SEG) == 0: raise RuntimeError('No hay corridas completas todavía.')
SEG = SEG.merge(APX, on=['n', 'id', 'rep', 'op'])
# J_kit con la normalización de B4 en el sustituto de diseño (por punto de operación)
refD = MET03[(MET03.planta == 'diseño') & (MET03.id == 'B4')].set_index('op')
has_ref = all(op in refD.index for op in SC['ops']) and all(SC[k] == sc03.get(k, SC[k]) for k in ['dr', 'dq2', 't_r', 't_d'])
if has_ref:
    SEG['J_normD'] = [COST['w_r'] * r.IAE_r / refD.loc[r.op].IAE_r + COST['w_d'] * r.IAE_d / refD.loc[r.op].IAE_d
                      + COST['w_u'] * r.TV / refD.loc[r.op].TV for r in SEG.itertuples()]
else:
    print('AVISO: puntos de operación o tramos distintos de NB03: J_kit y ΔJ no se calculan'); SEG['J_normD'] = np.nan
SEG.to_csv(os.path.join(OUT_DIR, 'metricas_por_tramo.csv'), index=False)
RAW = pd.concat([SEG.assign(y=f'T1 filtrada ({METRIC_FC} Hz)'), pd.DataFrame(raw)], ignore_index=True)
RAW.to_csv(os.path.join(OUT_DIR, 'metricas_por_tramo_y_cruda.csv'), index=False)
print('Efecto del ruido y del filtro (media sobre tramos):')
print(RAW.groupby(['id', 'y'])[['IAE_r', 'IAE_d', 'ITAE_d']].mean().round(1).unstack('y').to_string())

SUM = ['IAE_r', 'ITAE_r', 'IAE_d', 'ITAE_d', 'TV', 'E_J', 'IAE_op', 'aprox_s']
RUN = SEG.groupby(['n', 'id', 'rep'])[SUM].sum().join(SEG.groupby(['n', 'id', 'rep'])[['J_normD', 'overshoot_pct', 'emax_d']].mean()).reset_index()
RUN.to_csv(os.path.join(OUT_DIR, 'metricas_por_corrida.csv'), index=False)
print(RUN.round(2).to_string(index=False))

## 7. Tabla 8 (`tab:exp_cl`), Tabla 9 (`tab:exp_transfer`) y pruebas pareadas

- **Tabla 8:** media ± desviación sobre repeticiones de las sumas por corrida; Ms y φm de NB03 sobre `MS_MODEL`
  (peor punto de operación).
- **Tabla 9:** ΔJ = (J_kit − J_sur)/J_sur, con J_sur = costo del controlador en **su** sustituto de NB03 (P, B1, B2: fPINN
  de diseño; B3: PINN entera) con la misma normalización. B1 se diseñó sobre el FFOPDT, que no tiene canal de
  perturbación; su J_sur se toma en la fPINN (indicarlo en el texto). Tiempo de sintonía y evaluaciones del costo de NB03.
- **Pruebas pareadas** (P frente a cada comparador, pareando por repetición = bloque): t de Student pareada sobre
  IAE_r, IAE_d, TV y J_kit. Con 3 repeticiones la potencia es baja: reportar también las diferencias medias.

In [ ]:
ms_rows = RES03[RES03.modelo.str.startswith(MS_MODEL)]
MSPM = ms_rows.set_index('id')[['Ms', 'PM']] if len(ms_rows) else pd.DataFrame(columns=['Ms', 'PM'])
J_SUR_PLANT = {k: 'diseño' for k in CTRL_IDS}
b3p = RES03[RES03.modelo.str.startswith('IntegerPINN')].planta.unique()
if len(b3p): J_SUR_PLANT['B3'] = b3p[0]
ops_ok = [op for op in SC['ops']]
def j_sur(cid):
    m = MET03[(MET03.planta == J_SUR_PLANT.get(cid, 'diseño')) & (MET03.id == cid) & (MET03.op.isin(ops_ok))]
    if not has_ref or len(m) != len(ops_ok): return np.nan
    return np.mean([COST['w_r'] * r.IAE_r / refD.loc[r.op].IAE_r + COST['w_d'] * r.IAE_d / refD.loc[r.op].IAE_d
                    + COST['w_u'] * r.TV / refD.loc[r.op].TV for r in m.itertuples()])

pm_ = lambda s: f'{s.mean():.0f} $\\pm$ {s.std(ddof=1):.0f}' if len(s) > 1 else f'{s.mean():.0f}'
pm1 = lambda s, d=1: (f'{s.mean():.{d}f} $\\pm$ {s.std(ddof=1):.{d}f}' if len(s) > 1 else f'{s.mean():.{d}f}')
T8, tex8, T9, tex9 = [], [], [], []
for cid in CTRL_IDS:
    g = RUN[RUN.id == cid]
    if len(g) == 0: continue
    ms, pm = (MSPM.loc[cid].Ms, MSPM.loc[cid].PM) if cid in MSPM.index else (np.nan, np.nan)
    T8.append(dict(id=cid, n_corridas=len(g), **{f'{c}_media': g[c].mean() for c in SUM},
                   **{f'{c}_std': g[c].std(ddof=1) for c in SUM}, Ms=ms, PM=pm))
    tex8.append(f'{cid:3s}& {pm_(g.IAE_r)} & {pm_(g.ITAE_r / 1e3)} & {pm_(g.IAE_d)} & {pm_(g.ITAE_d / 1e3)} & '
                f'{pm_(g.TV)} & {pm_(g.E_J)} & {ms:.2f} & {pm:.1f} \\\\')
    js = j_sur(cid); jk = g.J_normD
    c3 = C03['controllers'][cid]
    T9.append(dict(id=cid, J_sur=js, J_kit_media=jk.mean(), J_kit_std=jk.std(ddof=1), dJ_pct=100 * (jk.mean() - js) / js,
                   dJ_pct_std=100 * jk.std(ddof=1) / js, sustituto=J_SUR_PLANT.get(cid), tiempo_sintonia_s=c3.get('time_s'),
                   evaluaciones=c3.get('n_eval')))
    if cid != 'B4':
        tex9.append(f'{cid:3s}& {pm1(100 * (jk - js) / js)} & {c3.get("time_s", 0):.0f} & {c3.get("n_eval", 0):.0f} \\\\')
T8 = pd.DataFrame(T8); T9 = pd.DataFrame(T9)
T8.to_csv(os.path.join(OUT_DIR, 'tab_exp_cl.csv'), index=False); T9.to_csv(os.path.join(OUT_DIR, 'tab_exp_transfer.csv'), index=False)
open(os.path.join(OUT_DIR, 'tab_exp_cl_rows.tex'), 'w', encoding='utf-8').write(
    '% ITAE en 10^3 °C s^2; sumas sobre puntos de operación; media ± std sobre repeticiones\n' + '\n'.join(tex8) + '\n')
open(os.path.join(OUT_DIR, 'tab_exp_transfer_rows.tex'), 'w', encoding='utf-8').write('\n'.join(tex9) + '\n')
pd.set_option('display.width', 220)
print('Tabla 8\n', T8[['id', 'n_corridas'] + [f'{c}_media' for c in SUM[:6]] + ['Ms', 'PM']].round(1).to_string(index=False))
print('\nTabla 9\n', T9.round(2).to_string(index=False))

# Pruebas pareadas P vs comparador (pareo por repetición)
PT = []
if 'P' in CTRL_IDS:
    for cid in [c for c in CTRL_IDS if c != 'P']:
        a = RUN[RUN.id == 'P'].set_index('rep'); b = RUN[RUN.id == cid].set_index('rep'); reps = a.index.intersection(b.index)
        for col in ['IAE_r', 'IAE_d', 'TV', 'J_normD']:
            x, y = a.loc[reps, col].values, b.loc[reps, col].values
            p = stats.ttest_rel(x, y).pvalue if len(reps) >= 2 and np.all(np.isfinite(x - y)) else np.nan
            PT.append(dict(comparador=cid, metrica=col, n=len(reps), P_media=x.mean(), comp_media=y.mean(),
                           dif_media=(x - y).mean(), mejora_pct=100 * (y.mean() - x.mean()) / y.mean(), p_ttest_pareada=p))
PT = pd.DataFrame(PT); PT.to_csv(os.path.join(OUT_DIR, 'pruebas_pareadas.csv'), index=False)
if len(PT): print('\nP frente a comparadores (mejora_pct > 0: P mejor)\n', PT.round(3).to_string(index=False))

## 8. Figuras

In [ ]:
cols = dict(P='C3', B1='C1', B2='C4', B3='C2', B4='k', A1_PID='C0', A1_rule='C9', A2='C5')
# Fig. 1: corrida representativa (primera repetición disponible), ensayo en el punto central
op_c = SC['ops'][len(SC['ops']) // 2]
fig, ax = plt.subplots(3, 1, figsize=(9, 7.5), sharex=True)
for cid in CTRL_IDS:
    g = PLAN[(PLAN.id == cid) & PLAN.archivo.map(lambda f: os.path.exists(os.path.join(RUN_DIR, f)))]
    if len(g) == 0: continue
    D = pd.read_csv(os.path.join(RUN_DIR, g.archivo.iloc[0])); d = D[(D.fase == 'ensayo') & (D.op == op_c)]
    ax[0].plot(d.tau, d.T1, c=cols.get(cid), lw=1.2 if cid == 'P' else 0.8, label=cid)
    ax[1].plot(d.tau, d.Q1, c=cols.get(cid), lw=0.7); ax[2].plot(d.tau, d.T2, c=cols.get(cid), lw=0.8)
    if cid == CTRL_IDS[0]: ax[0].plot(d.tau, d.r, c='0.6', ls='--', lw=1, label='r')
for a in ax: a.axvline(SC['t_r'], c='0.8', lw=0.8)
ax[0].set_ylabel('T1 [°C]'); ax[1].set_ylabel('Q1 [%]'); ax[2].set_ylabel('T2 [°C]'); ax[2].set_xlabel('t desde el inicio del ensayo [s]')
ax[0].legend(ncol=6, fontsize=8)
ax[0].set_title(f'{HW}: T̄ = {op_c:.0f} °C, r +{SC["dr"]:.0f} °C en t = {T_STEP_R} s y Q2 +{SC["dq2"]:.0f} % en t = {SC["t_r"]} s')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig01_corrida_representativa.png'), dpi=150)
plt.savefig(os.path.join(OUT_DIR, 'fig01_corrida_representativa.pdf')); plt.show(); plt.close('all')

# Fig. 2: dispersión entre repeticiones
fig, ax = plt.subplots(1, 4, figsize=(13, 3.2))
for a, c in zip(ax, ['IAE_r', 'IAE_d', 'TV', 'J_normD']):
    for i, cid in enumerate(CTRL_IDS):
        v = RUN[RUN.id == cid][c]; a.scatter(np.full(len(v), i), v, c=cols.get(cid), s=18); a.plot([i - .25, i + .25], [v.mean()] * 2, c=cols.get(cid))
    a.set_xticks(range(len(CTRL_IDS))); a.set_xticklabels(CTRL_IDS); a.set_title(c if c != 'J_normD' else 'J_kit (norm. B4 sust.)')
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig02_repeticiones.png'), dpi=150); plt.show(); plt.close('all')

# Fig. 3: corrida completa de P (aproximaciones + ensayos)
g = PLAN[(PLAN.id == CTRL_IDS[0]) & PLAN.archivo.map(lambda f: os.path.exists(os.path.join(RUN_DIR, f)))]
if len(g):
    D = pd.read_csv(os.path.join(RUN_DIR, g.archivo.iloc[0]))
    fig, ax = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    ax[0].plot(D.t / 60, D.r, c='0.6', ls='--', lw=1, label='r'); ax[0].plot(D.t / 60, D.T1, c='C3', lw=0.9, label='T1')
    ax[0].plot(D.t / 60, D.T2, c='C0', lw=0.8, label='T2'); ax[1].plot(D.t / 60, D.Q1, c='C3', lw=0.6, label='Q1')
    ax[1].plot(D.t / 60, D.Q2, c='C0', lw=0.8, label='Q2')
    for a in ax: a.legend(fontsize=8, ncol=3)
    ax[0].set_ylabel('°C'); ax[1].set_ylabel('%'); ax[1].set_xlabel('t [min]'); ax[0].set_title(f'Corrida completa: {g.archivo.iloc[0]}')
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'fig03_corrida_completa.png'), dpi=150); plt.show(); plt.close('all')

## 9. Empaquetar

In [ ]:
meta.update(fecha_fin=datetime.datetime.now().isoformat(timespec='seconds'), corridas_completas=int(PLAN.hecha.sum()),
            corridas_plan=len(PLAN), cost=COST, ms_model=MS_MODEL, j_sur_plant=J_SUR_PLANT)
save_meta()
zname = f"NB04_{HW}_resultados_{datetime.date.today().strftime('%Y%m%d')}{'_quick' if MODE == 'quick' else ''}.zip"
with zipfile.ZipFile(zname, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files: z.write(os.path.join(root, fn))
print('Guardado:', os.path.abspath(zname), f'({os.path.getsize(zname) / 1e6:.2f} MB) |',
      f'{int(PLAN.hecha.sum())} de {len(PLAN)} corridas')

## 10. Notas

- **Mismo escenario que NB03** en cada ensayo (reinicio sin salto en el equilibrio, r +5 °C a los 10 s, Q2 +40 % a los
  900 s), así que J_kit y J_sur se comparan directamente (ΔJ, H3). La diferencia con NB03: la planta (kit), el ambiente
  real, la resolución del sensor y el PWM de 8 bits, y que el arranque de cada ensayo es "casi" equilibrio.
- **Ruido y TV(u).** Con Kd ≈ 450–550 y Tf = 5 s, un salto de resolución del sensor (0,032 °C) mueve Q1 ≈ 3 %; NB03 ya
  incluía ruido de 0,03 °C, por eso TV(u) es alto en todos. Se reporta tal cual; no cambiar `TF_OVERRIDE` en la
  campaña principal (cambiaría los controladores frente a NB03).
- **Orden aleatorio por bloques** (`plan_NB04.csv`): reparte la deriva del ambiente; las pruebas pareadas emparejan por
  bloque. Anotar la temperatura de la sala si cambia mucho entre sesiones (`metadatos_nb04.json` guarda T_AMB y la
  espera de equilibrio de cada corrida).
- **Seguridad:** T_MAX = 85 °C en cada muestra; `finally` apaga los calentadores; celda de apagado de emergencia.